# Experiment 3: Market Regime Filtered RL Stock Trading Agent
## S&P 500 Regime → Bull/Bear → Buy/Sell Only | 20/50 EMA + BB Squeeze | 1:3 Risk/Reward

### Strategy Overview

This experiment introduces a **market regime filter** based on S&P 500 index trend:

| S&P 500 Condition | Regime | Allowed Action |
|---|---|---|
| Price > 50 EMA AND > 150 EMA | **BULL** (66.8%) | BUY (long) only |
| Price < 50 EMA AND < 150 EMA | **BEAR** (19.8%) | SELL (short) only |
| Otherwise | **NEUTRAL** (13.3%) | HOLD only |

### Entry Signal (20/50 EMA Crossover + BB Squeeze)
- EMA 20 crosses above EMA 50 → bullish cross
- EMA 20 crosses below EMA 50 → bearish cross
- BB width < 5% AND at 6-month low → squeeze (volatility contraction)
- ADX > 15 → trending market

### Risk Management (1:3 RR)
- SL options: 2%, 3%, 5%, 7%, 10%
- TP = 3 × SL (1:3 risk/reward ratio)
- Max drawdown cap: 25%

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

sys.path.insert(0, '..')
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from experiment_3.src.indicators import compute_indicators, get_feature_columns
from experiment_3.src.market_regime import compute_sp500_regime, merge_regime_to_stocks, get_regime_stats
from experiment_3.src.trading_env import RegimeStockEnv

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.style.use('seaborn-v0_8-darkgrid')
print('Ready!')

## 1. S&P 500 Regime Analysis

The S&P 500 trend determines which side of the market we trade.

In [ ]:
sp_df = pd.read_csv('experiment_3/data/SP500_daily.csv', parse_dates=['Date'])
sp_df = compute_sp500_regime(sp_df)
stats = get_regime_stats(sp_df)

print(f'SPX data: {len(sp_df)} days | {sp_df["Date"].min().date()} to {sp_df["Date"].max().date()}')
print(f'Regime distribution:')
print(f'  BULL:    {stats["bull_pct"]:.1f}%')
print(f'  BEAR:    {stats["bear_pct"]:.1f}%')
print(f'  NEUTRAL: {stats["neutral_pct"]:.1f}%')

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

ax1.plot(sp_df['Date'], sp_df['Close'], 'k-', lw=1, alpha=0.7, label='SPX Close')
ax1.plot(sp_df['Date'], sp_df['ema_50'], 'b-', lw=0.8, alpha=0.6, label='50 EMA')
ax1.plot(sp_df['Date'], sp_df['ema_150'], 'r-', lw=0.8, alpha=0.6, label='150 EMA')

# Highlight regimes
bull = sp_df[sp_df['regime'] == 1]
bear = sp_df[sp_df['regime'] == -1]
ax1.fill_between(bull['Date'], bull['Close'].min(), bull['Close'].max(), alpha=0.1, color='green', label=f'BULL ({len(bull)} days)')
ax1.fill_between(bear['Date'], bear['Close'].min(), bear['Close'].max(), alpha=0.1, color='red', label=f'BEAR ({len(bear)} days)')
ax1.set_title('S&P 500 — Market Regime Detection (50/150 EMA)')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(alpha=0.3)

colors = sp_df['regime'].map({1: 'green', -1: 'red', 0: 'gray'})
ax2.bar(sp_df['Date'], 1, color=colors, width=1)
ax2.set_title('Regime Timeline (green=BULL, red=BEAR, gray=NEUTRAL)')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 2. Stock Data with Regime + Indicators

Each stock row gets the S&P 500 regime for that date, plus 20/50 EMA crossover and BB squeeze indicators.

In [ ]:
# Load and prepare
st_df = pd.read_parquet('experiment_3/data/stocks_daily.parquet')
st_df = merge_regime_to_stocks(st_df, sp_df)

# Compute indicators for one symbol
sample = st_df[st_df['Symbol'] == 'AAPL'].copy()
sample = compute_indicators(sample)
feature_cols = get_feature_columns(sample)

print(f'Features ({len(feature_cols)}): {feature_cols}')
print(f'Rows after indicators: {len(sample)}')
print(f'Regime counts: {sample["regime_label"].value_counts().to_dict()}')

In [ ]:
# Dashboard
fig, axes = plt.subplots(3, 2, figsize=(18, 14))
s = sample.tail(250)

ax = axes[0,0]
ax.plot(s['Date'], s['Close'], 'k-', lw=1, label='Close')
ax.plot(s['Date'], s['ema_20'], 'b-', lw=0.8, alpha=0.6, label='EMA 20')
ax.plot(s['Date'], s['ema_50'], 'orange', lw=0.8, alpha=0.6, label='EMA 50')
ax.fill_between(s['Date'], s['bb_lower'], s['bb_upper'], alpha=0.15, color='blue', label='BB (20,2)')
ax.set_title('AAPL — 20/50 EMA + Bollinger Bands')
ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = axes[0,1]
ax.plot(s['Date'], s['ema_20_50_spread'], 'g-', lw=1, label='EMA 20-50 Spread')
ax.fill_between(s['Date'], 0, s['ema_20_50_spread'], where=(s['ema_20_50_spread']>0), alpha=0.2, color='green')
ax.fill_between(s['Date'], 0, s['ema_20_50_spread'], where=(s['ema_20_50_spread']<0), alpha=0.2, color='red')
cross = s[s['ema_cross_up']==1]
ax.scatter(cross['Date'], cross['ema_20_50_spread'], color='lime', s=40, marker='^', label='Cross Up')
ax.set_title('EMA 20/50 Crossover Signal')
ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = axes[1,0]
ax.plot(s['Date'], s['bb_width']*100, 'b-', lw=1, label='BB Width (%)')
ax.axhline(y=5, color='r', ls='--', alpha=0.5, label='Squeeze threshold (5%)')
sqz = s[s['bb_squeeze']==1]
ax.scatter(sqz['Date'], sqz['bb_width']*100, color='red', s=30, marker='v', label=f'BB Squeeze (n={len(sqz)})')
ax.set_title('BB Squeeze Detection')
ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = axes[1,1]
ax.plot(s['Date'], s['rsi_14'], 'purple', lw=1, label='RSI (14)')
ax.plot(s['Date'], s['adx_14']*100, 'brown', lw=1, label='ADX (14)')
ax.axhline(y=70, color='r', ls='--', alpha=0.3); ax.axhline(y=30, color='g', ls='--', alpha=0.3)
ax.set_title('RSI & ADX')
ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = axes[2,0]
colors = s['regime'].map({1:'green', -1:'red', 0:'gray'})
ax.bar(s['Date'], 1, color=colors, width=1)
ax.set_title('Market Regime (green=BULL, red=BEAR, gray=NEUTRAL)')
ax.set_ylim(0,1)

ax = axes[2,1]
sig = s[s['combined_signal']==1]
ax.plot(s['Date'], s['Close'], 'k-', lw=0.8, alpha=0.5)
ax.scatter(sig['Date'], sig['Close'], color='lime', s=50, marker='^', edgecolors='black', lw=0.5, label=f'Signal (n={len(sig)})')
ax.set_title('Combined Entry Signals (Cross + Squeeze + ADX>15)')
ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('AAPL — Regime-Aware Strategy Dashboard', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Training Configuration

- **Data**: 25 stocks × 5 years daily + S&P 500 regime
- **Features**: 22 scale-invariant indicators
- **Actions**: HOLD, CLOSE, BUY(5 SL, 1:3 RR), SELL(5 SL, 1:3 RR) = 12 total
- **Regime filter**: BUY only in BULL, SELL only in BEAR
- **Algorithm**: PPO, MLP [256,256,128], 200k steps

In [ ]:
# Process all symbols
results = []
for sym in sorted(st_df['Symbol'].unique()):
    sdf = st_df[st_df['Symbol'] == sym].copy()
    if len(sdf) < 200: continue
    sdf = compute_indicators(sdf)
    sdf['Symbol'] = sym
    results.append(sdf)

df = pd.concat(results, ignore_index=True).sort_values(['Symbol','Date']).reset_index(drop=True)
fc = get_feature_columns(df)
print(f'Processed: {len(df)} rows, {df["Symbol"].nunique()} symbols, {len(fc)} features')

# Split
train_p, test_p = [], []
for sym in sorted(df['Symbol'].unique()):
    sd = df[df['Symbol']==sym].sort_values('Date')
    n = int(len(sd) * 0.7)
    train_p.append(sd.iloc[:n]); test_p.append(sd.iloc[n:])
train_df = pd.concat(train_p).reset_index(drop=True)
test_df = pd.concat(test_p).reset_index(drop=True)
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

In [ ]:
# Normalization
mean = train_df[fc].values.astype(np.float32).mean(axis=0)
std = train_df[fc].values.astype(np.float32).std(axis=0)
std[std == 0] = 1.0

# Train
SL_OPTS = [2, 3, 5, 7, 10]
env_fn = lambda: RegimeStockEnv(
    df=train_df, window_size=30, sl_options_pct=SL_OPTS,
    feature_columns=fc, feature_mean=mean, feature_std=std,
    random_start=True, min_episode_steps=200, episode_max_steps=1500,
    hold_reward_weight=0.02, trade_penalty_pct=0.1, time_penalty_pct=0.005,
    max_drawdown_pct=0.25, drawdown_penalty_weight=2.0,
)

model = PPO('MlpPolicy', DummyVecEnv([env_fn]), verbose=1,
    learning_rate=3e-4, n_steps=2048, batch_size=64, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
    policy_kwargs=dict(net_arch=dict(pi=[256,256,128], vf=[256,256,128])),
)
model.learn(total_timesteps=200_000)
os.makedirs('experiment_3/models', exist_ok=True)
model.save('experiment_3/models/exp3_best')
print('Model saved!')

## 4. Results

In [ ]:
results = json.load(open('experiment_3/results/summary.json'))

print('='*60)
print(f'{"Metric":<25} {"Train (IS)":>15} {"Test (OOS)":>15}')
print('='*60)
print(f'{"Final Equity":<25} ${results["train"]["final"]:>14,.2f} ${results["test"]["final"]:>14,.2f}')
print(f'{"Return":<25} {results["train"]["return_pct"]:>14.2f}% {results["test"]["return_pct"]:>14.2f}%')
print(f'{"Trades":<25} {results["train"]["trades"]["n"]:>15} {results["test"]["trades"]["n"]:>15}')
print(f'{"Win Rate":<25} {results["train"]["trades"]["wr"]:>14.1f}% {results["test"]["trades"]["wr"]:>14.1f}%')
print(f'{"Max DD":<25} {"":>15} {results["test"]["max_dd"]:>14.2f}%')
print(f'{"Sharpe":<25} {"":>15} {results["test"]["sharpe"]:>14.2f}')
print(f'{"SL Options":<25} {results["config"]["sl_opts"]}')
print(f'{"TP Options (3x)":<25} {results["config"]["tp_opts"]}')
print('='*60)

In [ ]:
# Plot from saved image
from IPython.display import Image
Image('experiment_3/results/equity_curves.png')

## 5. Analysis

### Regime Distribution (5 years)
- **BULL: 66.8%** — Market trending above both 50 and 150 EMA
- **BEAR: 19.8%** — Market below both EMAs
- **NEUTRAL: 13.3%** — Price between EMAs or transitioning

### Key Findings
1. **Regime filtering reduces noise**: The agent only trades in clear trend regimes
2. **Sparse reward problem**: With only ~20% bear regime, SELL actions have limited opportunity to learn
3. **0 trades issue**: The agent may have learned that HOLD is optimal given limited profitable entries

### Improvements
1. **Increase entropy coefficient** to encourage more exploration
2. **Add regime-specific rewards**: Extra bonus for trading in the correct direction
3. **Use expert demonstrations**: Pre-train with EMA crossover rule-based policy
4. **Curriculum learning**: Start with simpler rules, gradually add complexity
5. **Longer training**: 500k-1M steps with learning rate decay